<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2>
            <span style="color: #B30033;">▍</span>TRABAJO DE FIN DE MÁSTER
        </h2>
        <h1 style="margin-top: -10px;">MARCO DE TRABAJO DE RECOMENDACIÓN DE TALENTO PARA LA FORMACIÓN DE EQUIPOS EN PROYECTOS DE INGENIERÍA DEL SOFTWARE</h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;">
                    <p style="margin: 0; font-size: 1.4em; font-weight: bold;">
                        Profesores
                    </p>
                </div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="display: block; font-size: 1em; color: #D0D0D0;">
                        Aurora Vizcaíno Barceló
                    </span>
                    <span style="display: block; font-size: 1em; color: #D0D0D0; margin-top: 5px;">
                        Julio Alberto López Gómez
                    </span>
                </div>
            </div>
        </div>
    </div>
</div>

<div style="margin-top: 30px; padding: 20px; background-color: rgba(179, 0, 51, 0.05); border-left: 4px solid #B30033; border-radius: 4px;">
    <h3 style="margin-top: 0; color: #B30033;">🎯 Objetivo de esta Libreta (Fase 5: Demostración del Framework y Ranking)</h3>
    <p style="font-size: 1.1em; line-height: 1.6; color: #E0E0E0;">
        Esta libreta culmina el desarrollo del framework <strong>NetSTAFF</strong>. Su objetivo es emular un caso de uso real donde, dados los datos de un proyecto entrante y nuestra base de empleados, un <strong>motor de machine learning explicable</strong> evalúa la idoneidad de los candidatos utilizando métricas de distancia y las reglas de conocimiento extraídas previamente.
    </p>
    <ul style="font-size: 1.1em; line-height: 1.6; color: #D0D0D0; margin-bottom: 0;">
        <li><strong>Ingesta del Proyecto:</strong> Carga de un proyecto sintético definido mediante atributos reales (stack tecnológico, políticas de remoto, tamaño de equipo, etc.).</li>
        <li><strong>Ranking de Hard Skills (Competencias Duras):</strong> Evaluación de la idoneidad técnica midiendo la distancia del Coseno/Manhattan entre los requisitos del proyecto y las tecnologías del candidato (datos 2025).</li>
        <li><strong>Ranking de Soft Skills (Competencias Blandas):</strong> Evaluación del riesgo y la idoneidad actitudinal aplicando las reglas expertas y minadas sobre el perfil psicológico y laboral del candidato (datos 2019).</li>
        <li><strong>Justificación Metodológica:</strong> Se presentan dos rankings separados debido a la naturaleza de las fuentes de datos (años distintos), planteando su agregación ponderada como línea de trabajo futuro.</li>
    </ul>
</div>
<br/>

<div style="border-bottom: 2px solid #555555; padding-bottom: 25px; margin-bottom: 10px">
    <div style="display: flex; align-items: center; margin-bottom: 10px;">
        <span style="color: #B30033; font-size: 1.5em; margin-right: 10px;">▍</span>
        <h3 style="margin: 0; font-size: 1.4em; font-weight: bold">
            Realizado por Miguel Ángel Sánchez-Medina Camacho <span style="font-size: 0.7em; font-style: italic; font-weight: normal;"></span>
        </h3>
    </div>
</div>

### 5.1. Configuración y Selección del Proyecto a Evaluar

Cargamos las librerías necesarias y extraemos nuestra cartera de proyectos sintéticos.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
from zoneinfo import ZoneInfo
from datetime import datetime

sys.path.append(os.path.abspath(os.path.join('../..')))
from src.hard_skills import config as config_hard_skills
from src.soft_skills import config as config_soft_skills

with open(config_hard_skills.REGLAS_EXPERTAS_PATH, encoding='utf-8') as f:
    reglas_expertas = json.load(f)['reglas']

with open('../../data/raw/proyectos_sinteticos.json', encoding='utf-8') as f:
    proyectos = json.load(f)['proyectos']

reglas_soft_proyecto = [r for r in reglas_expertas if r['tipo'] == 'soft_parcial']
reglas_hard_proyecto = [r for r in reglas_expertas if r['tipo'] == 'hard_parcial']


df_candidatos_hard_skills = pd.read_csv(config_hard_skills.MATRIZ_TRANSACCIONAL_PATH, index_col='CandidateId').fillna(0).astype(int)
df_candidatos_soft_skills = pd.read_csv(config_soft_skills.MATRIZ_TRANSACCIONAL_PATH, index_col='CandidateId').fillna(0).astype(int)

paises_mapa = pd.read_csv(config_hard_skills.PAISES_MAPA_PATH, sep=';')


print(f"Candidatos hard skills (2025): {len(df_candidatos_hard_skills)}")
print(f"Candidatos soft skills (2019): {len(df_candidatos_soft_skills)}")
print(f"Proyectos sintéticos cargados: {len(proyectos)}")
print(f"Países cargados: {len(paises_mapa)} ")
print(f"Reglas soft_parcial evaluables con proyecto real: {len(reglas_soft_proyecto)}")
print(f"Reglas hard_parcial evaluables con proyecto real: {len(reglas_hard_proyecto)}")

Candidatos hard skills (2025): 35434
Candidatos soft skills (2019): 88778
Proyectos sintéticos cargados: 20
Países cargados: 169 
Reglas soft_parcial evaluables con proyecto real: 9
Reglas hard_parcial evaluables con proyecto real: 1


In [2]:
df_geo_hard = pd.read_csv(config_hard_skills.PROCESSED_EDA_PATH, index_col='CandidateId', usecols=['CandidateId', 'Country'])
df_geo_hard['Country_normalizado'] = df_geo_hard['Country'].replace(config_hard_skills.TRADUCCIONES)
df_geo_hard = df_geo_hard.merge(
    paises_mapa[['Pais_Original', 'Idiomas_Oficiales']],
    left_on='Country_normalizado', right_on='Pais_Original', how='left'
)
df_geo_hard.index = pd.read_csv(config_hard_skills.PROCESSED_EDA_PATH, usecols=['CandidateId'])['CandidateId']

### 5.2. Ranking de Hard Skills — distancia Manhattan y similitud Coseno

In [3]:
def detalle_industria(proyecto, config_hard_skills):
    key_encuesta = proyecto.get('industry_survey_key')
    resultado = None
    if key_encuesta:
        codigo_industria = config_hard_skills.INDUSTRY_MAPPING.get(key_encuesta)
        if codigo_industria:
            resultado = config_hard_skills.INDUSTRY_CODE_DETALLE_MAPPING.get(codigo_industria)
    return resultado


def obtener_roles(proyecto):
    keys_roles = proyecto.get('role_devtype_keys')
    resultado = None
    if keys_roles:
        resultado = [f'DevType_{key}' for key in keys_roles]
    return resultado

In [4]:
def mapeo_proyecto(tech_stack_required, mapeo=config_hard_skills.MAPEO_TECNOLOGIAS):
    col_mendibles = [mapeo[t] for t in tech_stack_required if mapeo.get(t)]
    no_medibles = [t for t in tech_stack_required if not mapeo.get(t)]
    return col_mendibles, no_medibles

def calculo_score_ponderado(df, pesos):
    col = list(pesos.keys())
    values = np.array(list(pesos.values()))
    return df[col].values @ values

def match_candidatos_idioma(df_matriz, proyecto, df_geo_hard):
    idioma_proyecto = proyecto.get('common_project_language')
    resultado = None
    if idioma_proyecto:
        idiomas_candidatos = df_geo_hard['Idiomas_Oficiales'].reindex(df_matriz.index).fillna('')
        resultado = idiomas_candidatos.str.contains(idioma_proyecto, case=False, na=False).astype(int).values
    return resultado

def vector_candidato_proyecto(proyecto, df_matriz):
    columnas_tech, no_medibles_tech = mapeo_proyecto(proyecto['tech_stack_required'])
    bloq_candidato = [df_matriz[columnas_tech].values]
    bloq_ideal = [np.ones(len(columnas_tech))]
    dimensiones = list(columnas_tech)
    no_medibles = list(no_medibles_tech)

    nivel_senior = calculo_score_ponderado(df_matriz, config_hard_skills.SENIORITY_PESOS)
    bloq_candidato.append(nivel_senior.reshape(-1, 1))
    bloq_ideal.append(np.array([proyecto['seniority_threshold']]))
    dimensiones.append('seniority')

    score_teletrabajo = calculo_score_ponderado(df_matriz, config_hard_skills.REMOTO_PESOS)
    bloq_candidato.append(score_teletrabajo.reshape(-1, 1))
    bloq_ideal.append(np.array([proyecto['remote_work_pct'] / 100]))
    dimensiones.append('remote_work_pct')

    col_industria = detalle_industria(proyecto, config_hard_skills)
    if col_industria:
        bloq_candidato.append(df_matriz[[col_industria]].values)
        bloq_ideal.append(np.array([1.0]))
        dimensiones.append('industry')
    else:
        no_medibles.append(f"industry:{proyecto['industry']}")

    col_roles = obtener_roles(proyecto)
    if col_roles:
        match_roles = df_matriz[col_roles].max(axis=1).values.reshape(-1, 1)
        bloq_candidato.append(match_roles)
        bloq_ideal.append(np.array([1.0]))
        dimensiones.append('role_title')
    else:
        no_medibles.append(f"role_title:{proyecto['role_title']}")

    match_idioma = match_candidatos_idioma(df_matriz, proyecto, df_geo_hard)
    if match_idioma is not None:
        bloq_candidato.append(match_idioma.reshape(-1, 1))
        bloq_ideal.append(np.array([1.0]))
        dimensiones.append('idioma_proyecto')
    else:
        no_medibles.append(f"idioma:{proyecto.get('common_project_language')}")

    matriz_candidatos = np.hstack(bloq_candidato)
    vector_ideal = np.concatenate(bloq_ideal)
    return matriz_candidatos, vector_ideal, dimensiones, no_medibles

In [5]:
def asignacion_proyecto_candidatos_hard(proyecto, df_matriz, top_n=5):
    matriz_candidatos_finales, vector_ideales, m, no_medibles = vector_candidato_proyecto(proyecto, df_matriz)
    distancia_manhattan = np.abs(matriz_candidatos_finales - vector_ideales).sum(axis=1)
    norma_candidato = np.linalg.norm(matriz_candidatos_finales, axis=1)
    norma_ideal = np.linalg.norm(vector_ideales)
    with np.errstate(invalid='ignore', divide='ignore'):
        similitud_coseno = np.where(
            norma_candidato > 0,
            (matriz_candidatos_finales @ vector_ideales) / (norma_candidato * norma_ideal),
            0.0,
        )

    df_ranking_final = df_matriz.copy()
    df_ranking_final['distancia_manhattan'] = distancia_manhattan
    df_ranking_final['similitud_coseno'] = similitud_coseno
    ranking_final = df_ranking_final.sort_values('distancia_manhattan').head(top_n)
    return {'project_id': proyecto['project_id'],'dimensiones_evaluadas': m,'no_medibles': no_medibles,'ranking': ranking_final[['distancia_manhattan', 'similitud_coseno']]}

In [6]:
resultado_prueba = asignacion_proyecto_candidatos_hard(proyectos[0], df_candidatos_hard_skills)

print(f"Proyecto: {resultado_prueba['project_id']}")
print(f"Dimensiones evaluadas: {resultado_prueba['dimensiones_evaluadas']}")
print(f"No medibles: {resultado_prueba['no_medibles']}")
resultado_prueba['ranking']

Proyecto: PRY-2026-001
Dimensiones evaluadas: ['LanguageHaveWorkedWith_Python', 'LanguageHaveWorkedWith_SQL', 'PlatformHaveWorkedWith_Amazon Web Services (AWS)', 'seniority', 'remote_work_pct', 'industry', 'role_title', 'idioma_proyecto']
No medibles: []


,distancia_manhattan,similitud_coseno
CandidateId,,
33137,0.15,0.999144
28561,0.30,0.996263
8521,0.30,0.996263
27022,0.35,0.995407
7279,0.45,0.987723


### 5.3. Ranking de Soft Skills — riesgo activado por las reglas `soft_parcial`

### 5.4. Demostración con un proyecto real